# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
import sys
sys.path.append(os.path.abspath('..'))


In [3]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [4]:
load_dotenv(dotenv_path='../.env')
load_dotenv()
print("Environment loaded.")


Environment loaded.


### VectorDB Instance

In [5]:
chroma_client = chromadb.PersistentClient(path="games_chromadb")
print("Chroma client initialized.")


Chroma client initialized.


### Collection

In [6]:
openai_key = os.getenv("OPENAI_API_KEY")
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=openai_key,
    model_name="text-embedding-3-small",
    api_base="https://openai.vocareum.com/v1/"
)
print("Embedding function loaded.")


Embedding function loaded.


In [7]:
try:
    chroma_client.delete_collection("udaplay")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)
print("Collection created.")


Collection created.


### Add documents

In [8]:
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # Form content text
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )
print("All games indexed successfully.")


All games indexed successfully.


In [9]:
# Query verification
results = collection.query(
    query_texts=["Who developed FIFA 21?"],
    n_results=2
)
print("Search Results:")
for doc in results['documents'][0]:
    print(doc)


Search Results:
[Xbox Series X|S] Halo Infinite (2021) - The latest installment in the Halo franchise, featuring Master Chief's return in a new open-world setting.
[Xbox One] Minecraft (2014) - A sandbox game that allows players to build and explore infinite worlds, fostering creativity and adventure.
